# HW 3 — NER Fine-tuning with Transformers

Fine-tune a transformer encoder for Named Entity Recognition on FactRuEval 2016 dataset.
Explore multiple improvement strategies: MLM pre-training, advanced masking, and synthetic data augmentation.

## Environment Setup

This notebook is designed to run on Google Colab with a GPU runtime.
Alternatively, run locally with `uv sync --group hw_3` from the repo root.

In [ ]:
%%capture
!pip install -q -U transformers datasets evaluate seqeval accelerate sentencepiece
!pip install -q -U scikit-learn seaborn matplotlib pandas tqdm

## Notebook Setup

In [ ]:
import os
import re
import gc
import random
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
from datasets import load_dataset, Dataset, DatasetDict, concatenate_datasets
import evaluate

from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    AutoModelForMaskedLM,
    DataCollatorForTokenClassification,
    DataCollatorForLanguageModeling,
    DataCollatorForWholeWordMask,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    pipeline,
)

from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

Fix random seeds for reproducibility and detect device:

In [ ]:
RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
torch.cuda.manual_seed_all(RANDOM_STATE)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# Base encoder model — DeepPavlov ruBERT, a 12-layer cased BERT trained on Russian Wikipedia + news.
# Cased variant is important for NER since capitalization carries entity signal.
MODEL_NAME = "DeepPavlov/rubert-base-cased"

# Collect results from all experiments for final comparison
all_results = []

## 1. Load FactRuEval 2016 Dataset (2 pts)

[FactRuEval 2016](https://github.com/dialogue-evaluation/factRuEval-2016) is a Russian NER evaluation dataset
from the Dialogue shared task. It contains news texts annotated with Person, Organization, Location, and LocOrg entities.

We load the [HuggingFace version](https://huggingface.co/datasets/gusevski/factrueval2016) which provides
pre-tokenized sequences with BIO tags.

In [ ]:
raw = load_dataset("gusevski/factrueval2016")
print(raw)
print()

# Explore structure of a single example
for split_name in raw:
    ex = raw[split_name][0]
    print(f"\n--- {split_name} ---")
    for key in ex:
        val = ex[key]
        if isinstance(val, list):
            print(f"  {key}: list[{len(val)}], first 5: {val[:5]}")
        else:
            print(f"  {key}: {val}")

### Build label mapping

The dataset provides integer `ner_tags` and string `ner_tags_str`. We derive the label vocabulary from the string tags:

In [ ]:
# Build label vocabulary from string tags
all_tag_strs = set()
for split_name in raw:
    for example in raw[split_name]:
        tags = example["ner_tags_str"]
        if isinstance(tags[0], list):
            for sent_tags in tags:
                all_tag_strs.update(sent_tags)
        else:
            all_tag_strs.update(tags)

# Sort labels: O first, then B-/I- pairs
label_names = sorted(all_tag_strs)
if "O" in label_names:
    label_names.remove("O")
    label_names = ["O"] + label_names

label2id = {l: i for i, l in enumerate(label_names)}
id2label = {i: l for i, l in enumerate(label_names)}
num_labels = len(label_names)

print(f"Labels ({num_labels}): {label_names}")

# Identify entity types (strip B-/I- prefix)
entity_types = sorted({l.split("-", 1)[1] for l in label_names if "-" in l})
print(f"Entity types: {entity_types}")

### Flatten into individual examples

The HuggingFace version stores each split as a single row with long token/tag sequences.
We split these into sentence-level chunks for training. We split at punctuation marks (`.`, `!`, `?`)
to keep natural sentence boundaries and avoid breaking entity spans:

In [ ]:
def flatten_split(dataset_split):
    """Flatten a dataset split into individual sentence-level examples."""
    all_tokens = []
    all_tags = []

    for example in dataset_split:
        tokens = example["tokens"]
        tags_str = example["ner_tags_str"]

        # Handle nested structure (list of lists) vs flat structure
        if isinstance(tokens[0], list):
            for sent_tokens, sent_tags in zip(tokens, tags_str):
                if len(sent_tokens) >= 3:
                    int_tags = [label2id.get(t, 0) for t in sent_tags]
                    all_tokens.append(sent_tokens)
                    all_tags.append(int_tags)
        else:
            # Flat sequence — split into sentences at punctuation
            current_tokens = []
            current_tags = []
            for token, tag_str in zip(tokens, tags_str):
                current_tokens.append(token)
                current_tags.append(label2id.get(tag_str, 0))
                if token in (".", "!", "?") and len(current_tokens) >= 3:
                    all_tokens.append(current_tokens)
                    all_tags.append(current_tags)
                    current_tokens = []
                    current_tags = []
            if len(current_tokens) >= 3:
                all_tokens.append(current_tokens)
                all_tags.append(current_tags)

    return Dataset.from_dict({"tokens": all_tokens, "ner_tags": all_tags})


ds = DatasetDict()
for split_name in raw:
    ds[split_name] = flatten_split(raw[split_name])

# If no validation split, create one from train
if "validation" not in ds:
    split = ds["train"].train_test_split(test_size=0.15, seed=RANDOM_STATE)
    ds["train"] = split["train"]
    ds["validation"] = split["test"]

print(ds)
print(f"\nExample: {ds['train'][0]}")

In [ ]:
# Label distribution in train
tag_counter = Counter()
for example in ds["train"]:
    for tag_id in example["ner_tags"]:
        tag_counter[id2label[tag_id]] += 1

tag_df = pd.DataFrame(
    [(tag, count) for tag, count in tag_counter.most_common()],
    columns=["Tag", "Count"],
)
tag_df["Fraction"] = tag_df["Count"] / tag_df["Count"].sum()
print(tag_df.to_string(index=False))

# Entity-level stats
entity_count = sum(v for k, v in tag_counter.items() if k.startswith("B-"))
print(f"\nTotal entity spans (B-tags): {entity_count}")
print(f"Total tokens: {tag_df['Count'].sum()}")

## 2. Data Preparation (3 pts)

We tokenize with the BERT tokenizer and align NER labels to subword tokens.
Standard approach: first subword of each word gets the gold label, continuation subwords get `-100` (ignored in loss).
Special tokens (`[CLS]`, `[SEP]`, `[PAD]`) also get `-100`.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [ ]:
MAX_LENGTH = 256


def tokenize_and_align_labels(examples):
    """Tokenize pre-split words and align NER labels to subword tokens."""
    tokenized = tokenizer(
        examples["tokens"],
        is_split_into_words=True,
        truncation=True,
        max_length=MAX_LENGTH,
    )

    aligned_labels = []
    for i in range(len(examples["tokens"])):
        word_ids = tokenized.word_ids(batch_index=i)
        gold = examples["ner_tags"][i]
        prev = None
        labels = []
        for w in word_ids:
            if w is None:
                labels.append(-100)
            elif w != prev:
                labels.append(gold[w])
            else:
                labels.append(-100)  # subword continuation
            prev = w
        aligned_labels.append(labels)

    tokenized["labels"] = aligned_labels
    return tokenized


tokenized_data = ds.map(tokenize_and_align_labels, batched=True)
print(tokenized_data)
print(f"\nTrain example keys: {list(tokenized_data['train'][0].keys())}")

## 3. Fine-tune NER Model (3 pts)

We use `DeepPavlov/rubert-base-cased` — a 12-layer, 768-hidden BERT model pre-trained on Russian Wikipedia and news.
The cased variant preserves capitalization, which is an important signal for NER (e.g., "Москва" vs "москва").

Training strategy (from the lecture):
1. First evaluate the randomly-initialized classification head (baseline before training)
2. Full fine-tuning with early stopping

In [ ]:
seqeval_metric = evaluate.load("seqeval")


def compute_metrics(eval_pred):
    """Compute seqeval NER metrics for HF Trainer."""
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    y_true, y_pred = [], []
    for p, l in zip(preds, labels):
        true_seq, pred_seq = [], []
        for pi, li in zip(p, l):
            if li == -100:
                continue
            true_seq.append(id2label[li])
            pred_seq.append(id2label[pi])
        y_true.append(true_seq)
        y_pred.append(pred_seq)

    res = seqeval_metric.compute(
        predictions=y_pred, references=y_true, zero_division=0
    )
    return {
        "f1": res["overall_f1"],
        "precision": res["overall_precision"],
        "recall": res["overall_recall"],
    }


def predictions_to_seqeval(pred_output):
    """Convert Trainer predictions to seqeval format."""
    logits = pred_output.predictions
    labels = pred_output.label_ids
    preds = np.argmax(logits, axis=-1)

    y_true, y_pred = [], []
    for p, l in zip(preds, labels):
        true_seq, pred_seq = [], []
        for pi, li in zip(p, l):
            if li == -100:
                continue
            true_seq.append(id2label[li])
            pred_seq.append(id2label[pi])
        y_true.append(true_seq)
        y_pred.append(pred_seq)
    return y_true, y_pred


def evaluate_ner(trainer, dataset_split, approach_name):
    """Run prediction on a split, print metrics, and store results."""
    pred_output = trainer.predict(dataset_split)
    metrics = pred_output.metrics
    y_true, y_pred = predictions_to_seqeval(pred_output)
    detailed = seqeval_metric.compute(
        predictions=y_pred, references=y_true, zero_division=0
    )

    result = {
        "approach": approach_name,
        "f1": detailed["overall_f1"],
        "precision": detailed["overall_precision"],
        "recall": detailed["overall_recall"],
    }
    all_results.append(result)

    print(f"\n=== {approach_name} ===")
    print(f"  F1:        {result['f1']:.4f}")
    print(f"  Precision: {result['precision']:.4f}")
    print(f"  Recall:    {result['recall']:.4f}")

    # Per-entity breakdown
    for etype in entity_types:
        if etype in detailed:
            e = detailed[etype]
            print(f"  {etype}: P={e['precision']:.3f} R={e['recall']:.3f} F1={e['f1']:.3f} ({e['number']} entities)")

    return result

### Baseline — before fine-tuning

Evaluate the model with a randomly initialized classification head to establish a lower bound:

In [ ]:
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer, padding="longest")

model_untrained = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
)

trainer_untrained = Trainer(
    model=model_untrained,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

test_split = tokenized_data.get("test", tokenized_data["validation"])
evaluate_ner(trainer_untrained, test_split, "Before fine-tuning")

del model_untrained, trainer_untrained
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

### Fine-tune the encoder

Full fine-tuning with AdamW optimizer, linear warmup, and early stopping.
- **learning_rate=2e-5** — standard for BERT fine-tuning (from the original BERT paper)
- **warmup_ratio=0.1** — gradual warmup prevents catastrophic forgetting
- **weight_decay=0.01** — L2 regularization to prevent overfitting on a small dataset

In [ ]:
def train_ner(
    model_name_or_path,
    train_dataset,
    eval_dataset,
    output_dir,
    num_epochs=10,
    learning_rate=2e-5,
    batch_size=16,
):
    """Train a token classification model and return the trainer."""
    model = AutoModelForTokenClassification.from_pretrained(
        model_name_or_path,
        num_labels=num_labels,
        id2label=id2label,
        label2id=label2id,
        ignore_mismatched_sizes=True,
    )

    args = TrainingArguments(
        output_dir=output_dir,
        learning_rate=learning_rate,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size * 2,
        num_train_epochs=num_epochs,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        fp16=torch.cuda.is_available(),
        report_to="none",
        save_total_limit=2,
        seed=RANDOM_STATE,
        warmup_ratio=0.1,
        weight_decay=0.01,
        max_grad_norm=1.0,
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
    )

    trainer.train()
    return trainer

In [ ]:
trainer_baseline = train_ner(
    MODEL_NAME,
    tokenized_data["train"],
    tokenized_data["validation"],
    output_dir="ner_baseline",
)

evaluate_ner(trainer_baseline, test_split, "Baseline NER")

In [ ]:
del trainer_baseline
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 4. MLM Pre-training + NER (5 pts)

Domain-adaptive pre-training: before NER fine-tuning, we continue MLM pre-training on the FactRuEval texts.
This adapts the model's language representations to the target domain (Russian news),
potentially improving downstream NER performance.

We use standard random token masking with 15% probability (following the original BERT recipe).
The model starts from the original `DeepPavlov/rubert-base-cased` checkpoint.

In [ ]:
# Prepare texts for MLM — join word tokens back into strings
mlm_texts = [" ".join(tokens) for tokens in ds["train"]["tokens"]]
print(f"MLM texts: {len(mlm_texts)}")
print(f"Example: {mlm_texts[0][:200]}...")


def tokenize_for_mlm(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        return_special_tokens_mask=True,
    )


mlm_dataset = Dataset.from_dict({"text": mlm_texts})
mlm_tokenized = mlm_dataset.map(
    tokenize_for_mlm, batched=True, remove_columns=["text"]
)
print(f"MLM tokenized examples: {len(mlm_tokenized)}")

In [ ]:
def train_mlm(
    model_name_or_path,
    train_dataset,
    mlm_collator,
    output_dir,
    num_epochs=10,
    learning_rate=5e-5,
    batch_size=16,
):
    """Pre-train model with MLM and return path to saved checkpoint."""
    model = AutoModelForMaskedLM.from_pretrained(model_name_or_path)

    args = TrainingArguments(
        output_dir=output_dir,
        learning_rate=learning_rate,
        per_device_train_batch_size=batch_size,
        num_train_epochs=num_epochs,
        fp16=torch.cuda.is_available(),
        report_to="none",
        save_strategy="epoch",
        save_total_limit=1,
        seed=RANDOM_STATE,
        warmup_ratio=0.1,
        weight_decay=0.01,
        logging_steps=50,
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        data_collator=mlm_collator,
    )

    trainer.train()

    save_path = os.path.join(output_dir, "final")
    trainer.save_model(save_path)
    tokenizer.save_pretrained(save_path)

    del model, trainer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return save_path

In [ ]:
# Standard random masking MLM
mlm_collator_random = DataCollatorForLanguageModeling(
    tokenizer=tokenizer, mlm=True, mlm_probability=0.15
)

mlm_random_path = train_mlm(
    MODEL_NAME,
    mlm_tokenized,
    mlm_collator_random,
    output_dir="mlm_random",
)
print(f"MLM (random masking) saved to: {mlm_random_path}")

In [ ]:
# Fine-tune for NER from MLM-adapted checkpoint
trainer_mlm_ner = train_ner(
    mlm_random_path,
    tokenized_data["train"],
    tokenized_data["validation"],
    output_dir="ner_after_mlm_random",
)

evaluate_ner(trainer_mlm_ner, test_split, "MLM (random) → NER")

del trainer_mlm_ner
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 5. Advanced Masking Strategies (5 pts)

We try two improved masking strategies for MLM pre-training,
both starting from the original `DeepPavlov/rubert-base-cased` checkpoint:

1. **Whole Word Masking (WWM)** — instead of masking individual subword tokens,
   mask all subwords of a word together. This prevents the model from trivially
   predicting masked subwords from unmasked parts of the same word.

2. **Concept Masking** — preferentially mask NER entity tokens.
   This forces the model to learn stronger contextual representations
   of entity positions, directly benefiting downstream NER.

### 5.1 Whole Word Masking

In [ ]:
# DataCollatorForWholeWordMask groups subword tokens belonging to the same word
# and masks them together. For BERT (WordPiece), it identifies continuations
# by the "##" prefix.
wwm_collator = DataCollatorForWholeWordMask(
    tokenizer=tokenizer, mlm=True, mlm_probability=0.15
)

mlm_wwm_path = train_mlm(
    MODEL_NAME,
    mlm_tokenized,
    wwm_collator,
    output_dir="mlm_wwm",
)
print(f"MLM (WWM) saved to: {mlm_wwm_path}")

In [ ]:
# Fine-tune for NER from WWM-adapted checkpoint
trainer_wwm_ner = train_ner(
    mlm_wwm_path,
    tokenized_data["train"],
    tokenized_data["validation"],
    output_dir="ner_after_mlm_wwm",
)

evaluate_ner(trainer_wwm_ner, test_split, "MLM (WWM) → NER")

del trainer_wwm_ner
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

### 5.2 Concept Masking (NER Entity Masking)

Instead of uniform random masking, we preferentially mask tokens that belong to NER entities.
This teaches the model to predict entity tokens from their surrounding context,
which is exactly what NER needs — understanding what context signals the presence of an entity.

Implementation: entity tokens are masked with 80% probability, non-entity tokens with ~5%
(to maintain some general language modeling signal).

In [ ]:
def prepare_concept_mlm_data(examples):
    """Tokenize texts and mark entity positions for concept masking."""
    tokenized = tokenizer(
        examples["tokens"],
        is_split_into_words=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_special_tokens_mask=True,
    )

    all_entity_masks = []
    for i in range(len(examples["tokens"])):
        word_ids = tokenized.word_ids(batch_index=i)
        tags = examples["ner_tags"][i]
        entity_mask = []
        for wid in word_ids:
            if wid is not None and wid < len(tags) and tags[wid] != 0:
                entity_mask.append(True)
            else:
                entity_mask.append(False)
        all_entity_masks.append(entity_mask)

    tokenized["entity_mask"] = all_entity_masks
    return tokenized


concept_mlm_data = ds["train"].map(
    prepare_concept_mlm_data, batched=True, remove_columns=ds["train"].column_names
)

# Check entity token ratio
entity_token_count = sum(sum(m) for m in concept_mlm_data["entity_mask"])
total_token_count = sum(len(m) for m in concept_mlm_data["entity_mask"])
print(f"Entity tokens: {entity_token_count}/{total_token_count} ({entity_token_count / total_token_count:.1%})")

In [ ]:
class ConceptMaskingCollator:
    """MLM data collator that preferentially masks NER entity tokens.

    Entity tokens are masked with `entity_prob` probability,
    non-entity tokens with `other_prob` probability.
    Follows the standard 80/10/10 masking protocol (replace/random/keep).
    """

    def __init__(self, tokenizer, entity_prob=0.80, other_prob=0.05):
        self.tokenizer = tokenizer
        self.entity_prob = entity_prob
        self.other_prob = other_prob
        self.mask_token_id = tokenizer.convert_tokens_to_ids(tokenizer.mask_token)

    def __call__(self, features):
        entity_masks = [f.pop("entity_mask") for f in features]

        batch = self.tokenizer.pad(
            features, return_tensors="pt", padding="longest"
        )

        labels = batch["input_ids"].clone()

        # Build probability matrix: high prob for entities, low for others
        special_mask = torch.tensor(
            [
                self.tokenizer.get_special_tokens_mask(ids.tolist(), already_has_special_tokens=True)
                for ids in batch["input_ids"]
            ],
            dtype=torch.bool,
        )

        entity_tensor = torch.zeros_like(labels, dtype=torch.bool)
        for i, em in enumerate(entity_masks):
            length = min(len(em), labels.size(1))
            for j in range(length):
                if em[j]:
                    entity_tensor[i, j] = True

        prob_matrix = torch.where(
            entity_tensor,
            torch.full_like(labels, self.entity_prob, dtype=torch.float),
            torch.full_like(labels, self.other_prob, dtype=torch.float),
        )
        prob_matrix.masked_fill_(special_mask, 0.0)

        masked_indices = torch.bernoulli(prob_matrix).bool()
        labels[~masked_indices] = -100

        # 80% [MASK], 10% random, 10% keep
        indices_replaced = torch.bernoulli(torch.full(labels.shape, 0.8)).bool() & masked_indices
        batch["input_ids"][indices_replaced] = self.mask_token_id

        indices_random = (
            torch.bernoulli(torch.full(labels.shape, 0.5)).bool()
            & masked_indices
            & ~indices_replaced
        )
        random_words = torch.randint(len(self.tokenizer), labels.shape, dtype=torch.long)
        batch["input_ids"][indices_random] = random_words[indices_random]

        batch["labels"] = labels
        # Remove special_tokens_mask if present (not needed by model)
        batch.pop("special_tokens_mask", None)
        return batch

In [ ]:
concept_collator = ConceptMaskingCollator(tokenizer, entity_prob=0.80, other_prob=0.05)

mlm_concept_path = train_mlm(
    MODEL_NAME,
    concept_mlm_data,
    concept_collator,
    output_dir="mlm_concept",
)
print(f"MLM (concept masking) saved to: {mlm_concept_path}")

In [ ]:
# Fine-tune for NER from concept-masking-adapted checkpoint
trainer_concept_ner = train_ner(
    mlm_concept_path,
    tokenized_data["train"],
    tokenized_data["validation"],
    output_dir="ner_after_mlm_concept",
)

evaluate_ner(trainer_concept_ner, test_split, "MLM (concept) → NER")

del trainer_concept_ner
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 6. Synthetic Data Augmentation (5 pts)

Generate synthetic NER annotations on a large Russian text corpus using a strong teacher model,
then combine with the original FactRuEval training data.

- **Teacher model**: `Davlan/xlm-roberta-large-ner-hrl` — a 560M-parameter XLM-RoBERTa
  fine-tuned on HRL NER benchmark across multiple languages including Russian.
  Much larger than our 110M-param encoder, so its predictions serve as reliable pseudo-labels.

- **Corpus**: [Lenta.ru news](https://huggingface.co/datasets/IlyaGusev/lenta_ru) — Russian news articles,
  domain-relevant for FactRuEval. We use 10,000 texts as recommended.

- **Label mapping**: the teacher model outputs PER/ORG/LOC/DATE labels;
  we keep PER/ORG/LOC (matching FactRuEval types) and ignore DATE.

In [ ]:
# Load the teacher NER model
teacher_model_name = "Davlan/xlm-roberta-large-ner-hrl"
ner_pipeline = pipeline(
    "ner",
    model=teacher_model_name,
    aggregation_strategy="simple",
    device=0 if torch.cuda.is_available() else -1,
    batch_size=8,
)
print(f"Teacher model loaded: {teacher_model_name}")

In [ ]:
# Load Lenta.ru corpus
SYNTHETIC_SIZE = 10_000
lenta = load_dataset("IlyaGusev/lenta_ru", split=f"train[:{SYNTHETIC_SIZE}]")
print(f"Loaded {len(lenta)} texts from Lenta.ru")
print(f"Columns: {lenta.column_names}")
print(f"Example text: {lenta[0]['text'][:200]}...")

In [ ]:
# Map teacher model entity types to factrueval types
# Detect factrueval type naming convention from label_names
def build_type_mapping():
    """Map pipeline output types (PER/ORG/LOC) to factrueval label types."""
    factrueval_types = {l.split("-", 1)[1] for l in label_names if "-" in l}

    mapping = {}
    for ft in factrueval_types:
        ft_upper = ft.upper()
        if ft_upper in ("PER", "PERSON"):
            mapping["PER"] = ft
        elif ft_upper in ("ORG", "ORGANIZATION"):
            mapping["ORG"] = ft
        elif ft_upper in ("LOC", "LOCATION"):
            mapping["LOC"] = ft
    return mapping


type_mapping = build_type_mapping()
print(f"Type mapping (pipeline → factrueval): {type_mapping}")


def entities_to_bio(words, word_offsets, entities):
    """Convert pipeline entity spans to BIO tags aligned with word tokens."""
    tags = ["O"] * len(words)

    for ent in entities:
        ent_type = ent["entity_group"]
        if ent_type not in type_mapping:
            continue
        mapped_type = type_mapping[ent_type]

        start_char, end_char = ent["start"], ent["end"]

        first_word = True
        for i, (ws, we) in enumerate(word_offsets):
            if ws < end_char and we > start_char:  # overlap
                if first_word:
                    tags[i] = f"B-{mapped_type}"
                    first_word = False
                else:
                    tags[i] = f"I-{mapped_type}"

    return tags


def text_to_words_and_offsets(text):
    """Split text into words and compute character offsets."""
    words = []
    offsets = []
    for match in re.finditer(r"\S+", text):
        words.append(match.group())
        offsets.append((match.start(), match.end()))
    return words, offsets

In [ ]:
# Generate synthetic NER annotations
synthetic_tokens = []
synthetic_tags = []
MAX_TEXT_LEN = 2000  # truncate very long texts for pipeline efficiency

texts = [t[:MAX_TEXT_LEN] for t in lenta["text"] if len(t.strip()) >= 20]
print(f"Processing {len(texts)} texts through teacher model...")

for i in tqdm(range(0, len(texts), 32)):
    batch_texts = texts[i : i + 32]
    try:
        batch_results = ner_pipeline(batch_texts)
    except Exception as e:
        print(f"Error at batch {i}: {e}")
        continue

    for text, entities in zip(batch_texts, batch_results):
        words, offsets = text_to_words_and_offsets(text)
        if len(words) < 5:
            continue
        bio_tags = entities_to_bio(words, offsets, entities)
        int_tags = [label2id.get(t, 0) for t in bio_tags]
        synthetic_tokens.append(words)
        synthetic_tags.append(int_tags)

print(f"\nGenerated {len(synthetic_tokens)} synthetic examples")
entity_count = sum(1 for tags in synthetic_tags for t in tags if id2label.get(t, "O").startswith("B-"))
print(f"Total entity spans in synthetic data: {entity_count}")

In [ ]:
# Free teacher model memory
del ner_pipeline
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
# Create combined dataset: factrueval + synthetic
synthetic_ds = Dataset.from_dict(
    {"tokens": synthetic_tokens, "ner_tags": synthetic_tags}
)

# Tokenize synthetic data
synthetic_tokenized = synthetic_ds.map(tokenize_and_align_labels, batched=True)

# Concatenate with original training data
combined_train = concatenate_datasets(
    [tokenized_data["train"], synthetic_tokenized]
).shuffle(seed=RANDOM_STATE)

print(f"Original train:  {len(tokenized_data['train'])} examples")
print(f"Synthetic:       {len(synthetic_tokenized)} examples")
print(f"Combined train:  {len(combined_train)} examples")

In [ ]:
# Fine-tune on combined data (from original checkpoint)
trainer_synthetic = train_ner(
    MODEL_NAME,
    combined_train,
    tokenized_data["validation"],
    output_dir="ner_synthetic",
)

evaluate_ner(trainer_synthetic, test_split, "Synthetic data + NER")

del trainer_synthetic
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 7. Results Comparison (3 pts)

In [ ]:
results_df = pd.DataFrame(all_results)
results_df = results_df.sort_values("f1", ascending=False).reset_index(drop=True)
print(results_df.to_string(index=False))

In [ ]:
# Visualization
fig, ax = plt.subplots(figsize=(10, 5))

# Exclude "Before fine-tuning" for cleaner chart
plot_df = results_df[results_df["approach"] != "Before fine-tuning"].copy()

x = np.arange(len(plot_df))
width = 0.25

ax.bar(x - width, plot_df["precision"], width, label="Precision")
ax.bar(x, plot_df["f1"], width, label="F1")
ax.bar(x + width, plot_df["recall"], width, label="Recall")

ax.set_xlabel("Approach")
ax.set_ylabel("Score")
ax.set_title("NER Performance Comparison")
ax.set_xticks(x)
ax.set_xticklabels(plot_df["approach"], rotation=30, ha="right")
ax.legend()
ax.set_ylim(0, 1.0)

for i, row in enumerate(plot_df.itertuples()):
    ax.text(i, row.f1 + 0.02, f"{row.f1:.3f}", ha="center", fontsize=8)

plt.tight_layout()
plt.show()

## 8. Conclusions

*(Fill in after running experiments)*